# 문제 6

(kaggle 형 문제) **prob4**를 바탕으로 아래 데이터셋을 만든다.

. **prob6_train**: DateHour 변수 기준으로 2021년 8월 14일 전(8월 14일 미포함) 데이터 (행의 수: 5256개)

. **prob6_test**: DateHour 변수 기준으로 2021년 8월 14일 이후(8월 14일 포함) 데이터 (행의 수: 744 개)

일 때, **prob6_train**으로 target을 예측하는 모델을 만들어, 

**prob6_test**의 target에 대한 MAE를 최소화하는 모델을 만든다. 

$MAE(Y, \hat{Y})=\frac{1}{n}\sum^n_{i=1}|y_i - \hat{y_i}|$ 

**prob6_test**의 예측 결과를 아래와 같은 형식으로 출력한다. 파일명은 answer6.csv 이다.

|DateHour|TotalHour|
|--------|---------|
|2021-08-14 00:00:00|102.607580|
|2021-08-14 01:00:00|94.078890|
....

※ 실제 상황에 가까운 연습을 위해 **prob6_test**의 target 값은 모른다는 상황을 가정하고 문제를 푼다.

# Kaggle형 풀이 단계

Step 0: Kaggle용 데이터셋을 만든다.

Step 1: 검증 방법을 정하고, 검증 루틴을 만듭니다. metric: MAE

Step 2: Baseline 모델을 만듭니다

Step 3: 모델 선택 루틴을 만듭니다.

|DateHour|	TotalHour|
|----|----|
|2021-08-14 00:00:00|	102.607580|
|2021-08-14 01:00:00|	94.078890|
....	

Step 4: 모델 개선 작업을 합니다.



## 데이터 전처리

In [16]:
# 실행 환경 확인

import pandas as pd
import numpy as np
import sklearn
import scipy
import statsmodels
#import mlxtend
import sys
#import xgboost as xgb

print(sys.version)
for i in [pd, np, sklearn, scipy, statsmodels]:
    print(i.__name__, i.__version__)

3.7.4 (tags/v3.7.4:e09359112e, Jul  8 2019, 20:34:20) [MSC v.1916 64 bit (AMD64)]
pandas 0.25.1
numpy 1.18.5
sklearn 0.21.3
scipy 1.5.2
statsmodels 0.11.1


In [17]:
df_elec = pd.read_csv('elec.csv', parse_dates=['Date', 'DateHour'])
df_info = pd.read_csv('info.csv')
df_info['Date'] = pd.to_datetime(df_info['Date'])

In [18]:
df_elec1 = df_elec.pivot(
    index='DateHour', 
    columns='Minute', 
    values='Value').reset_index() # index에 위치한 DateHour를 컬럼에 위치시킵니다
df_elec1

Minute,DateHour,15분,30분,45분,60분
0,2021-01-01 00:00:00,62,61,61,61
1,2021-01-01 01:00:00,96,93,116,113
2,2021-01-01 02:00:00,106,96,106,107
3,2021-01-01 03:00:00,92,110,110,109
4,2021-01-01 04:00:00,108,105,106,108
...,...,...,...,...,...
6163,2021-09-14 19:00:00,152,151,171,139
6164,2021-09-14 20:00:00,124,130,128,130
6165,2021-09-14 21:00:00,134,130,125,124
6166,2021-09-14 22:00:00,100,109,120,114


In [19]:
holi = pd.to_datetime(["2021-01-01", "2021-02-11", "2021-02-12", "2021-03-01", "2021-05-05", "2021-05-19", "2021-08-16"]).date

# pd.Series.dt accessor를 통해 파생 변수들을 만듭니다.
df_elec1 = df_elec1.assign(
    DayName = df_elec1['DateHour'].dt.weekday, 
    Hour = df_elec1['DateHour'].dt.hour,
    AM = (df_elec1['DateHour'].dt.hour >= 12).astype('int'),
    Weekend_yn = df_elec1['DateHour'].dt.weekday.isin([5, 6]).astype('int'),
    Holiday_yn = df_elec1['DateHour'].dt.date.isin(holi).astype('int'), # df_elec1['DateHour'].dt.date는 pd.Series 입니다.
    Avg = df_elec1[['15분', '30분', '45분', '60분']].mean(axis=1),
    TotalHour = df_elec1[['15분', '30분', '45분', '60분']].sum(axis=1),
)

In [20]:
df_info1 = df_info.fillna(0)

In [21]:
df_basetable1 = df_elec1.merge(
    df_info1, 
    left_on='DateHour', 
    right_on='Date', how='inner'
).drop(columns='Date') # DateHour가 컬럼에 있으므로 DateHour, Date 모두 존재하여, Date는 삭제했습니다.
display(df_basetable1.head())
df_basetable1.shape

,DateHour,15분,30분,45분,60분,DayName,Hour,AM,Weekend_yn,Holiday_yn,Avg,TotalHour,생산량,기온,풍속,습도,강수량,전기요금(계절),공장인원,인건비
0,2021-01-01 00:00:00,62,61,61,61,4,0,0,0,1,61.25,245,0,-3.2,2.4,71,0.0,109.8,0.0,1.5
1,2021-01-01 01:00:00,96,93,116,113,4,1,0,0,1,104.50,418,0,-4.5,1.5,77,0.0,109.8,0.0,1.5
2,2021-01-01 02:00:00,106,96,106,107,4,2,0,0,1,103.75,415,0,-3.9,2.6,58,0.0,109.8,0.0,1.5
3,2021-01-01 03:00:00,92,110,110,109,4,3,0,0,1,105.25,421,0,-4.1,2.6,56,0.0,109.8,0.0,1.5
4,2021-01-01 04:00:00,108,105,106,108,4,4,0,0,1,106.75,427,0,-4.6,2.6,60,0.0,109.8,0.0,1.5


(6168, 20)

In [22]:
# shift를 이용합니다.
df_prob3 = df_basetable1.sort_values('DateHour').reset_index(drop=True)
df_prob3['target'] = df_prob3['TotalHour'].shift(-24)
df_prob4 = pd.concat(
    [df_prob3] + [
        df_prob3['TotalHour'].shift(24 * i).rename('lag_{}'.format(i)) for i in range(1, 7)
    ], axis=1
).dropna()
df_prob4.head()

,DateHour,15분,30분,45분,60분,DayName,Hour,AM,Weekend_yn,Holiday_yn,...,전기요금(계절),공장인원,인건비,target,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6
144,2021-01-07 00:00:00,22,22,22,25,3,0,0,0,0,...,109.8,0.000000,1.5,252.0,96.0,96.0,271.0,88.0,253.0,245.0
145,2021-01-07 01:00:00,26,22,22,22,3,1,0,0,0,...,109.8,0.369565,1.5,396.0,85.0,85.0,432.0,99.0,418.0,418.0
146,2021-01-07 02:00:00,22,21,21,22,3,2,0,0,0,...,109.8,0.000000,1.5,411.0,85.0,85.0,439.0,88.0,415.0,415.0
147,2021-01-07 03:00:00,22,22,22,23,3,3,0,0,0,...,109.8,0.629213,1.5,398.0,90.0,90.0,426.0,88.0,421.0,421.0
148,2021-01-07 04:00:00,22,26,26,23,3,4,0,0,0,...,109.8,2.865979,1.5,420.0,93.0,93.0,435.0,95.0,427.0,427.0


In [23]:
# shift를 사용하지 않고 target를 구하는 방법입니다. (값을 구하기만 하고, 따로 prob4로 만들지는 않았습니다.)
df_basetable1.join(
    df_basetable1[['DateHour','TotalHour']].assign(
        DateHour=lambda x: x['DateHour'] - pd.Timedelta(days=1)
    ).set_index('DateHour').rename(columns={'TotalHour': 'target'}),
    on='DateHour', how='inner'
)

,DateHour,15분,30분,45분,60분,DayName,Hour,AM,Weekend_yn,Holiday_yn,...,TotalHour,생산량,기온,풍속,습도,강수량,전기요금(계절),공장인원,인건비,target
0,2021-01-01 00:00:00,62,61,61,61,4,0,0,0,1,...,245,0,-3.2,2.4,71,0.0,109.8,0.000000,1.5,253
1,2021-01-01 01:00:00,96,93,116,113,4,1,0,0,1,...,418,0,-4.5,1.5,77,0.0,109.8,0.000000,1.5,418
2,2021-01-01 02:00:00,106,96,106,107,4,2,0,0,1,...,415,0,-3.9,2.6,58,0.0,109.8,0.000000,1.5,415
3,2021-01-01 03:00:00,92,110,110,109,4,3,0,0,1,...,421,0,-4.1,2.6,56,0.0,109.8,0.000000,1.5,421
4,2021-01-01 04:00:00,108,105,106,108,4,4,0,0,1,...,427,0,-4.6,2.6,60,0.0,109.8,0.000000,1.5,427
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6139,2021-09-13 19:00:00,162,160,148,122,0,19,1,0,0,...,592,2820,22.3,2.1,78,0.0,167.2,4.763514,1.5,613
6140,2021-09-13 20:00:00,113,122,122,126,0,20,1,0,0,...,483,14,22.2,1.3,78,0.0,167.2,0.028986,1.5,512
6141,2021-09-13 21:00:00,122,131,122,115,0,21,1,0,0,...,490,72,21.9,1.9,77,0.0,167.2,0.146939,1.5,513
6142,2021-09-13 22:00:00,97,108,122,113,0,22,1,0,0,...,440,11,21.7,1.4,77,0.0,167.2,0.025000,1.5,443


In [24]:
# 문제에서 제시한 데이터셋을 만듭니다.
df_prob6_train = df_prob4.loc[df_prob4['DateHour'] < pd.to_datetime('2021-08-14 00:00:00')]
df_prob6_test = df_prob4.loc[df_prob4['DateHour'] >= pd.to_datetime('2021-08-14 00:00:00')]
# 정답 target은 떼어냅니다.
s_kaggle_ans =  df_prob6_test.pop('target') # Kaggle의 정답으로 간주하겠습니다, 실제 시험에서는 공개 되지 않습니다.

## Step 1: 검증 방법을 정하고, 검증 루틴을 만듭니다.

- 평가 방법: 실제 ML 모델이 투입이 되었을 경우의 상황을 고려하여 설계합니다
> 이 Task의 구성: 지금까지 누적된 데이터를 가지고 미래의 TotalHour를 예측하는 모델으 만듭니다. 
>
>  모델의 학습주기: 1개월 
  
  
- 검증 방법 정하기
> 분리 방법: Random, 계층적(Stratfied), 그룹(동일한 그룹 분리가 되지 않게), 시점 기준 분리, ...
>
> 진행 방법
>   * Hold-out (train / valid) => 결과의 편차가 크지 않은 경우
>
>   * cross-validation => n개의 셋으로 나누어, 하나씩 학습제외하고 검증셋 지정하여 n번의 검증을 통해 평균으로 성능 측정
>
>   * Monte Carlo => Hold-out 또는 cv 셋의 구성 달리하며 n번 수행 하는 것 


- ML에 있어 핵심 입니다.

In [25]:
df_prob6_test['DateHour'].agg(['min', 'max'])

min   2021-08-14 00:00:00
max   2021-09-13 23:00:00
Name: DateHour, dtype: datetime64[ns]

In [34]:
X_all = df_prob6_test.columns.tolist()
np.array(X_all)

array(['DateHour', '15분', '30분', '45분', '60분', 'DayName', 'Hour', 'AM',
       'Weekend_yn', 'Holiday_yn', 'Avg', 'TotalHour', '생산량', '기온', '풍속',
       '습도', '강수량', '전기요금(계절)', '공장인원', '인건비', 'lag_1', 'lag_2', 'lag_3',
       'lag_4', 'lag_5', 'lag_6'], dtype='<U10')

In [28]:
df_train = df_prob6_train.loc[df_prob6_train['DateHour'] < pd.to_datetime('2021-07-14 00:00:00')]
df_valid = df_prob6_train.loc[df_prob6_train['DateHour'] >= pd.to_datetime('2021-07-14 00:00:00')]

In [29]:
df_valid['DateHour'].agg(['min', 'max'])

min   2021-07-14 00:00:00
max   2021-08-13 23:00:00
Name: DateHour, dtype: datetime64[ns]

## Step2: Baseline Model 모델을 만듭니다.

**Idea**

- 모델
> SVR - kernel - 'rbf', C = 10, gamma = 0.1
  
- 전처리 
> 표준화: 'TotalHour', 'lag_1' ~ 'lag_6' 
>
> 가변수화: 'DayName'
>
>통과: 'AM'

In [60]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline

from sklearn.svm import SVR

ct = ColumnTransformer([
    ('std', StandardScaler(), ['TotalHour'] + ['lag_{}'.format(i) for i in range(1, 7)]),
    ('ohe', OneHotEncoder(categories='auto'), ['DayName']),
    ('pt', 'passthrough', ['AM'])
])

reg_svr = make_pipeline(
    ct, SVR(kernel = 'rbf', C = 10, gamma = 0.1)
)
reg_svr.fit(df_train[X_all], df_train['target'])

Pipeline(memory=None,
         steps=[('columntransformer',
                 ColumnTransformer(n_jobs=None, remainder='drop',
                                   sparse_threshold=0.3,
                                   transformer_weights=None,
                                   transformers=[('std',
                                                  StandardScaler(copy=True,
                                                                 with_mean=True,
                                                                 with_std=True),
                                                  ['TotalHour', 'lag_1',
                                                   'lag_2', 'lag_3', 'lag_4',
                                                   'lag_5', 'lag_6']),
                                                 ('ohe',
                                                  OneHotEncoder(categorical_features=None,
                                                                categories='auto',
        

In [58]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(
    df_valid['target'], reg_svr.predict(df_valid[X_all])
)

145.34693972591637

In [59]:
# Kaggle 루틴의 원리의 이해에 도움을 드리기 위해
# 동일한 과정을 ColumnTransformer와 Pipeline없이 구성해봅니다. 
# 검증 과정에 해당하는 것이고, 
# 선택 과정은 이와 같은 과정으로 데이터를 바꿔서 수행 루틴을 구성해야 합니다.
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, OneHotEncoder

std_scaler = StandardScaler()
ohe = OneHotEncoder(categories='auto', sparse = False)
X_std= ['TotalHour'] + ['lag_{}'.format(i) for i in range(1, 7)]
X_ohe = ['DayName']
X_pt = ['AM']
std_scaler.fit(df_train[X_std])
ohe.fit(df_train[X_ohe])

reg_svr_direct = SVR(kernel = 'rbf', C = 10, gamma = 0.1)
reg_svr_direct.fit(
    pd.concat([
        pd.DataFrame(std_scaler.transform(df_train[X_std]), index = df_train.index),
        pd.DataFrame(ohe.transform(df_train[X_ohe]), index =df_train.index),
        df_train[X_pt]
    ], axis=1), df_train['target']
)
prd = reg_svr_direct.predict(
    pd.concat([
        pd.DataFrame(std_scaler.transform(df_valid[X_std]), index = df_valid.index),
        pd.DataFrame(ohe.transform(df_valid[X_ohe]), index = df_valid.index),
        df_valid[X_pt]
    ], axis=1)
)

print("Without ColumnTransformer & Pipeline", mean_absolute_error(df_valid['target'], prd)) # 145.34693972591637

Without ColumnTransformer & Pipeline 145.34693972591637


## Step3: 모델 선택

**prob6_test**의 예측 결과를 아래와 같은 형식으로 출력한다. 파일명은 answer6.csv 이다.

|DateHour|TotalHour|
|--------|---------|
|2021-08-14 00:00:00|102.607580|
|2021-08-14 01:00:00|94.078890|

In [63]:
reg_svr.fit(df_prob6_train[X_all], df_prob6_train['target'])
prd = reg_svr.predict(df_prob6_test[X_all])
prd

array([ 80.88583003,  84.99642555,  90.73510936,  85.21184197,
        79.06669589,  80.84785986,  70.96165824,  51.42532078,
       125.41861586, 110.09505354, 103.53626616, 114.55107513,
        94.31011692, 104.95120092, 113.15932455,  93.74058225,
       106.4983882 ,  98.8500524 ,  88.31366552,  74.60278888,
        71.52830305,  83.31550006,  83.74107278,  79.91465722,
       117.00830223,  98.99707673,  83.73885314,  87.27788215,
        84.71961563, 100.16264481,  79.28361259, 280.67557272,
       675.78483488, 692.54417476, 685.28278835, 678.47007208,
       520.35423355, 676.01874191, 677.36093218, 672.1553248 ,
       686.62387569, 607.56351054, 452.77917557, 429.42464101,
       458.10203117, 461.79261398, 440.67331093, 380.76699066,
       255.90302787, 378.06880866, 384.52973608, 390.31102492,
       397.77935678, 317.24210188, 390.58944612, 510.88737554,
       659.71235076, 681.76698248, 659.63270342, 665.74417254,
       435.74549263, 654.95937218, 636.98219927, 620.51

In [67]:
pd.DataFrame({
    'DateHour': df_prob6_test['DateHour'],
    'TotalHour': prd
}).to_csv('answer6.csv', index = None)